In [13]:
# =============================================================================
# SYSTEMATIC ERROR REDUCTION ANALYSIS
# One-vs-Rest Brain Region Classification
# =============================================================================

try:
    import numpy as np
    import pandas as pd
    import json
    from pathlib import Path
    import warnings
    import re

    # Visualization
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    import plotly.express as px

    # Machine Learning
    from sklearn.metrics import confusion_matrix

    # Configuration
    warnings.filterwarnings('ignore')
    pd.set_option('display.max_columns', None)
    pd.set_option('display.precision', 2)

    # Color Palette for Visualizations
    COLORS = {
        'baseline': '#e74c3c',      # Red
        'step1': '#f39c12',          # Orange  
        'step2': '#3498db',          # Blue
        'step3': '#2ecc71',          # Green
        'same_h_same_n': '#1A5276',
        'same_h_diff_n': '#1D8348',
        'diff_h_same_n': '#7D3C98',
        'diff_h_diff_n': '#2E4053'
    }
    print("Imports successful.")
except Exception as e:
    print(f"Import Error: {e}")

Imports successful.


In [14]:
# =============================================================================
# PATH VERIFICATION
# =============================================================================


BASE_PATH = Path('/home/sjoon/projects/brain_connectivity_classifier/data')
RESULTS_DIR = BASE_PATH / 'results'

# Template for repetitive paths
def res_p(sub, task): return RESULTS_DIR / sub / task
f_sub = 'full_connectivity_analysis'
l_sub, r_sub = 'hemisphere_analysis/left_hemisphere', 'hemisphere_analysis/right_hemisphere'
ovr, t_ovr = 'one_vs_rest', 'task_testing_one_vs_rest'

PATHS = {
    'full_cv': res_p(f_sub, ovr) / 'cv_summary.json',
    'full_task': res_p(f_sub, t_ovr) / 'task_testing_summary.json',
    'full_pred': res_p(f_sub, t_ovr) / 'task_predictions.npy',
    'full_true': res_p(f_sub, t_ovr) / 'task_true_labels.npy',
    
    'left_cv': res_p(l_sub, ovr) / 'cv_summary.json',
    'left_task': res_p(l_sub, t_ovr) / 'task_testing_summary.json',
    'left_pred': res_p(l_sub, t_ovr) / 'task_predictions.npy',
    'left_true': res_p(l_sub, t_ovr) / 'task_true_labels.npy',
    
    'right_cv': res_p(r_sub, ovr) / 'cv_summary.json',
    'right_task': res_p(r_sub, t_ovr) / 'task_testing_summary.json',
    'right_pred': res_p(r_sub, t_ovr) / 'task_predictions.npy',
    'right_true': res_p(r_sub, t_ovr) / 'task_true_labels.npy',
    
    'full_regions': BASE_PATH / 'FULL_region_info.csv',
    'lh_regions': BASE_PATH / 'LH_region_info.csv',
    'rh_regions': BASE_PATH / 'RH_region_info.csv',
    'coordinates': BASE_PATH / 'network_files' / 'Schaefer2018_200Parcels_Tian_32Parcels.csv'
}

# Condensed Verification
print("Path Verification:")
missing_paths = [n for n, p in PATHS.items() if not (print(f"  {'✓' if p.exists() else '✗'} {n}{'' if p.exists() else ' - MISSING'}") or p.exists())]

if missing_paths: print(f"\n⚠ Warning: {len(missing_paths)} path(s) missing")
else: print("\n✓ All paths verified successfully")

Path Verification:
  ✓ full_cv
  ✓ full_task
  ✓ full_pred
  ✓ full_true
  ✓ left_cv
  ✓ left_task
  ✓ left_pred
  ✓ left_true
  ✓ right_cv
  ✓ right_task
  ✓ right_pred
  ✓ right_true
  ✓ full_regions
  ✓ lh_regions
  ✓ rh_regions
  ✓ coordinates

✓ All paths verified successfully


In [15]:
# =============================================================================
# DATA LOADING
# =============================================================================
print("Loading data...\n")

# 1. Load region metadata
full_region_info = pd.read_csv(PATHS['full_regions'])
lh_region_info = pd.read_csv(PATHS['lh_regions'])
rh_region_info = pd.read_csv(PATHS['rh_regions'])
coords_df = pd.read_csv(PATHS['coordinates'])

print(f"✓ Metadata loaded: Full ({len(full_region_info)}), Left ({len(lh_region_info)}), Right ({len(rh_region_info)})")

# 2. Load performance summaries
def load_summary(path):
    with open(path, 'r') as f: return json.load(f)

s_keys = ['full_cv', 'full_task', 'left_cv', 'left_task', 'right_cv', 'right_task']
full_cv, full_task, left_cv, left_task, right_cv, right_task = [load_summary(PATHS[k]) for k in s_keys]

# 3. Load predictions and labels
p_keys = [('full_pred', 'full_true'), ('left_pred', 'left_true'), ('right_pred', 'right_true')]
preds_data = [ (np.load(PATHS[p], allow_pickle=True), np.load(PATHS[t], allow_pickle=True)) for p, t in p_keys]

(full_preds, full_true), (left_preds, left_true), (right_preds, right_true) = preds_data



print(f"\n✓ Summaries and Predictions loaded:")
print(f"  Full: {len(full_preds):,} | Left: {len(left_preds):,} | Right: {len(right_preds):,}")
print(f"\n DATA LOADING COMPLETE")

Loading data...

✓ Metadata loaded: Full (232), Left (116), Right (116)

✓ Summaries and Predictions loaded:
  Full: 46,400 | Left: 23,200 | Right: 23,200

 DATA LOADING COMPLETE


In [16]:
# =============================================================================
# BASELINE MODEL PERFORMANCE (Full Connectivity - 232 Regions)
# =============================================================================

def calculate_metrics(cv_data, task_data, predictions, true_labels):
    # Extract fold metrics and calculate means
    v_accs = [f['val_accuracy'] for f in cv_data['fold_metrics']]
    t_accs = [f['train_accuracy'] for f in cv_data['fold_metrics']]
    
    cv_t, cv_v = np.mean(t_accs) * 100, np.mean(v_accs) * 100
    task_a = task_data['task_test_accuracy'] * 100
    t_errs = np.sum(predictions != true_labels)
    t_samp = len(predictions)
    
    return {
        'cv_train_acc': cv_t, 'cv_val_acc': cv_v, 'generalization_gap': cv_t - cv_v,
        'task_acc': task_a, 'accuracy_drop': cv_v - task_a,
        'total_samples': t_samp, 'total_errors': t_errs, 'error_rate': (t_errs / t_samp) * 100
    }

# Calculate and Print
m = calculate_metrics(full_cv, full_task, full_preds, full_true)
baseline_metrics = m # Keep original name for downstream use

print(f"{'='*80}\nBASELINE MODEL PERFORMANCE (Full Connectivity)\n{'='*80}")
print(f"\nCV Performance:\n  Train: {m['cv_train_acc']:.2f}% | Val: {m['cv_val_acc']:.2f}% | Gap: {m['generalization_gap']:.2f}%")
print(f"\nTask Performance:\n  Test:  {m['task_acc']:.2f}% | Drop: {m['accuracy_drop']:.2f}%")
print(f"\nError Analysis:\n  Samples: {m['total_samples']:,} | Errors: {m['total_errors']:,} | Rate: {m['error_rate']:.2f}%")

BASELINE MODEL PERFORMANCE (Full Connectivity)

CV Performance:
  Train: 82.20% | Val: 76.24% | Gap: 5.96%

Task Performance:
  Test:  73.04% | Drop: 3.21%

Error Analysis:
  Samples: 46,400 | Errors: 12,511 | Rate: 26.96%


In [17]:
# =============================================================================
# BASELINE ERROR BREAKDOWN ANALYSIS
# Categorize errors by hemisphere and network agreement
# =============================================================================

def analyze_error_breakdown(predictions, true_labels, region_info):
    # Create error dataframe and join metadata
    meta = region_info.set_index('region_idx')[['hemisphere', 'network']]
    df = pd.DataFrame({'true': true_labels, 'pred': predictions})
    df = df[df['true'] != df['pred']].join(meta.add_prefix('true_'), on='true').join(meta.add_prefix('pred_'), on='pred')
    
    # Logic masks
    s_h, s_n = df['true_hemisphere'] == df['pred_hemisphere'], df['true_network'] == df['pred_network']
    tot = len(df)
    
    breakdown = {
        'total_errors': tot,
        'same_h_same_n': len(df[s_h & s_n]), 'same_h_diff_n': len(df[s_h & ~s_n]),
        'diff_h_same_n': len(df[~s_h & s_n]), 'diff_h_diff_n': len(df[~s_h & ~s_n])
    }
    
    if tot > 0:
        for k in list(breakdown.keys())[1:]:
            breakdown[f'{k}_pct'] = (breakdown[k] / tot) * 100
            
    return breakdown, df

# Analyze and Print
baseline_breakdown, baseline_errors_df = analyze_error_breakdown(full_preds, full_true, full_region_info)
b = baseline_breakdown
cross_err = b['diff_h_same_n'] + b['diff_h_diff_n']

print(f"{'='*80}\nBASELINE ERROR BREAKDOWN\n{'='*80}\nTotal Errors: {b['total_errors']:,}\n")
for k, lab in [('same_h_same_n', 'Same H / Same N'), ('same_h_diff_n', 'Same H / Diff N'), 
               ('diff_h_same_n', 'Diff H / Same N'), ('diff_h_diff_n', 'Diff H / Diff N')]:
    print(f"{lab:<36} {b[k]:4,} ({b[k+'_pct']:5.1f}%)")

print(f"\n{'='*80}\nKEY INSIGHT: Cross-Hemisphere Errors\n{'='*80}")
print(f"Total Cross-Hemisphere Errors: {cross_err:,} ({cross_err/b['total_errors']*100:.1f}%)\n→ Step 2: Hemisphere-Specific Models")

BASELINE ERROR BREAKDOWN
Total Errors: 12,511

Same H / Same N                      2,056 ( 16.4%)
Same H / Diff N                      5,001 ( 40.0%)
Diff H / Same N                      2,613 ( 20.9%)
Diff H / Diff N                      2,841 ( 22.7%)

KEY INSIGHT: Cross-Hemisphere Errors
Total Cross-Hemisphere Errors: 5,454 (43.6%)
→ Step 2: Hemisphere-Specific Models


In [18]:
# =============================================================================
# STEP 2: HEMISPHERE-SPECIFIC MODELS 
# =============================================================================

# Calculate metrics and breakdowns for both hemispheres
hemi_data = []
for cv, task, preds, true, info in [
    (left_cv, left_task, left_preds, left_true, lh_region_info),
    (right_cv, right_task, right_preds, right_true, rh_region_info)
]:
    hemi_data.append((calculate_metrics(cv, task, preds, true), analyze_error_breakdown(preds, true, info)[0]))

(left_metrics, left_breakdown), (right_metrics, right_breakdown) = hemi_data

# Combined performance calculations
combined_errors = left_breakdown['total_errors'] + right_breakdown['total_errors']
combined_samples = left_metrics['total_samples'] + right_metrics['total_samples']
combined_accuracy = ((combined_samples - combined_errors) / combined_samples) * 100
error_red = baseline_breakdown['total_errors'] - combined_errors

print(f"{'='*80}\nSTEP 2: HEMISPHERE-SPECIFIC MODELS\n{'='*80}")
for label, m, b in [("Left", left_metrics, left_breakdown), ("Right", right_metrics, right_breakdown)]:
    print(f"\n{label} Hemisphere:\n  Val Acc: {m['cv_val_acc']:.2f}% | Test Acc: {m['task_acc']:.2f}% | Errors: {b['total_errors']:,}")

print(f"\nCombined Performance:\n  Accuracy: {combined_accuracy:.2f}% | Total Errors: {combined_errors:,}")

print(f"\n{'='*80}\nERROR REDUCTION (Baseline → Hemisphere Models)\n{'='*80}")
print(f"Baseline Errors:   {baseline_breakdown['total_errors']:,}")
print(f"Errors Reduced:    {error_red:,} ({error_red/baseline_breakdown['total_errors']*100:.2f}%)")
print(f"Accuracy Gain:     {combined_accuracy - baseline_metrics['task_acc']:.2f}%")

STEP 2: HEMISPHERE-SPECIFIC MODELS

Left Hemisphere:
  Val Acc: 92.79% | Test Acc: 88.75% | Errors: 2,611

Right Hemisphere:
  Val Acc: 92.48% | Test Acc: 87.97% | Errors: 2,791

Combined Performance:
  Accuracy: 88.36% | Total Errors: 5,402

ERROR REDUCTION (Baseline → Hemisphere Models)
Baseline Errors:   12,511
Errors Reduced:    7,109 (56.82%)
Accuracy Gain:     15.32%


In [19]:
# =============================================================================
# PERFORMANCE COMPARISON TABLE 
# =============================================================================
models = {
    'Baseline (Full)': (baseline_metrics, baseline_breakdown),
    'Left Hemisphere': (left_metrics, left_breakdown),
    'Right Hemisphere': (right_metrics, right_breakdown)
}

comparison_data = []
for name, (m, b) in models.items():
    comparison_data.append({
        'Model': name,
        'CV Val Acc (%)': f"{m['cv_val_acc']:.2f}",
        'Test Acc (%)': f"{m['task_acc']:.2f}",
        'Total Errors': b['total_errors'],
        'Error Rate (%)': f"{m['error_rate']:.2f}",
        **{k: f"{b[v]} ({b[v+'_pct']:.1f}%)" for k, v in [
            ('Same H/Same N', 'same_h_same_n'), ('Same H/Diff N', 'same_h_diff_n'),
            ('Diff H/Same N', 'diff_h_same_n'), ('Diff H/Diff N', 'diff_h_diff_n')]}
    })

comparison_df = pd.DataFrame(comparison_data)
print(f"{'='*80}\nCOMPREHENSIVE MODEL COMPARISON\n{'='*80}\n{comparison_df.to_string(index=False)}")

COMPREHENSIVE MODEL COMPARISON
           Model CV Val Acc (%) Test Acc (%)  Total Errors Error Rate (%) Same H/Same N Same H/Diff N Diff H/Same N Diff H/Diff N
 Baseline (Full)          76.24        73.04         12511          26.96  2056 (16.4%)  5001 (40.0%)  2613 (20.9%)  2841 (22.7%)
 Left Hemisphere          92.79        88.75          2611          11.25    168 (6.4%)  2443 (93.6%)      0 (0.0%)      0 (0.0%)
Right Hemisphere          92.48        87.97          2791          12.03    163 (5.8%)  2628 (94.2%)      0 (0.0%)      0 (0.0%)


In [20]:
# =============================================================================
# STEP 3: SPATIAL DISTANCE FILTERING (20mm Threshold) - BILATERAL
# =============================================================================
DISTANCE_THRESHOLD = 20 

# Load coordinates and clean ROI Names
coords_df = pd.read_csv('/home/sjoon/projects/brain_connectivity_classifier/data/network_files/Schaefer2018_200Parcels_Tian_32Parcels.csv')
coords_df['ROI Name'] = coords_df['ROI Name'].str.replace('17Networks_', '')

# Compact coordinate mapping
coords_map = {row['ROI Name']: {'R': row['R'], 'A': row['A'], 'S': row['S']} for _, row in coords_df.iterrows()}

def calculate_distance_mask_by_name(regions_list, coords_map, threshold):
    num_regions = len(regions_list)
    dist_mask, missing_coords = np.ones((num_regions, num_regions), dtype=bool), []
    
    for i in range(num_regions):
        for j in range(num_regions):
            r1, r2 = regions_list[i], regions_list[j]
            if r1 in coords_map and r2 in coords_map:
                p1 = np.array([coords_map[r1]['R'], coords_map[r1]['A'], coords_map[r1]['S']])
                p2 = np.array([coords_map[r2]['R'], coords_map[r2]['A'], coords_map[r2]['S']])
                if np.linalg.norm(p1 - p2) < threshold: dist_mask[i, j] = False
            else:
                dist_mask[i, j] = False
                for r in [r1, r2]:
                    if r not in coords_map and r not in missing_coords: missing_coords.append(r)
    return dist_mask, missing_coords

# Process Both Hemispheres in a Loop
print("Calculating distance masks...")
hemi_data = [('LEFT', '/home/sjoon/projects/brain_connectivity_classifier/data/LH_region_info.csv'),
             ('RIGHT', '/home/sjoon/projects/brain_connectivity_classifier/data/RH_region_info.csv')]

results = []
for label, path in hemi_data:
    reg_names = pd.read_csv(path)['region_name'].tolist()
    mask, missing = calculate_distance_mask_by_name(reg_names, coords_map, DISTANCE_THRESHOLD)
    results.append((mask, missing))
    
    print(f"\n✓ {label} HEMISPHERE: {mask.shape}\n  Distal: {mask.sum():,} ({mask.sum()/mask.size*100:.1f}%)"
          f"\n  Nearby: {(~mask).sum():,} ({(~mask).sum()/mask.size*100:.1f}%)")
    if missing: print(f"  Missing coords: {len(missing)} regions")

# Re-assign to your original variable names
lh_dist_mask, lh_missing = results[0]
rh_dist_mask, rh_missing = results[1]

Calculating distance masks...

✓ LEFT HEMISPHERE: (116, 116)
  Distal: 12,952 (96.3%)
  Nearby: 504 (3.7%)

✓ RIGHT HEMISPHERE: (116, 116)
  Distal: 12,940 (96.2%)
  Nearby: 516 (3.8%)


In [21]:
from sklearn.metrics import confusion_matrix

# =============================================================================
# 1. CORE SPATIAL FILTERING & ERROR CALCULATION
# =============================================================================

def get_confusion_matrix(predictions, true_labels, num_regions=116):
    region_indices = np.arange(0, num_regions)
    cm = confusion_matrix(true_labels, predictions, labels=region_indices)
    np.fill_diagonal(cm, 0) 
    return cm

# Generate Error Matrices & Apply Distance Masks
cm_left, cm_right = get_confusion_matrix(left_preds, left_true), get_confusion_matrix(right_preds, right_true)
cm_left_distal, cm_right_distal = cm_left * lh_dist_mask, cm_right * rh_dist_mask

# Define Anatomical Groups
CORTICAL_INDICES = list(range(0, 100))
SUBCORTICAL_INDICES = list(range(100, 116))

# =============================================================================
# 2. THE TWO-PERSPECTIVE ANALYTICS
# =============================================================================

results = {"LH": {"cm": cm_left, "distal": cm_left_distal}, 
           "RH": {"cm": cm_right, "distal": cm_right_distal}}

print(f"{'='*100}\nSTEP 3: INTEGRATED SPATIAL & ANATOMICAL ANALYSIS\n{'='*100}")

# Helper for percentage calculation to keep code small
pct = lambda part, total: (part / total * 100) if total > 0 else 0

for side, data in results.items():
    cm, dist = data["cm"], data["distal"]
    total_err = cm.sum()
    dist_err = dist.sum()
    near_err = total_err - dist_err
    
    print(f"\n[{side} HEMISPHERE] Total Errors: {int(total_err):,} | Reduction: {pct(near_err, total_err):.2f}%")
    print("-" * 100)
    print(f"{'Metric':<35} {'Nearby':>10} {'Distal':>10} {'Total':>10} {'Nearby %':>12}")
    
    # Perspective 1: Origin Analysis (How many errors started here?)
    for name, idx in [("From Cortical", CORTICAL_INDICES), ("From Subcortical", SUBCORTICAL_INDICES)]:
        t = cm[idx, :].sum()
        d = dist[idx, :].sum()
        n = t - d
        print(f"{name:<35} {int(n):>10,} {int(d):>10,} {int(t):>10,} {pct(n, t):>11.2f}%")

    # Perspective 2: Confusion Type (What boundaries were crossed?)
    confusions = [
        ("Within-Cortical (C<->C)", (CORTICAL_INDICES, CORTICAL_INDICES)),
        ("Within-Subcortical (S<->S)", (SUBCORTICAL_INDICES, SUBCORTICAL_INDICES)),
        ("Cross-Group (C<->S)", "cross")
    ]
    
    print(f"\n{'CONFUSION TYPES (Boundary Analysis)':<35}")
    for label, idx_pair in confusions:
        if idx_pair == "cross":
            t = (cm[np.ix_(CORTICAL_INDICES, SUBCORTICAL_INDICES)].sum() + 
                 cm[np.ix_(SUBCORTICAL_INDICES, CORTICAL_INDICES)].sum())
            d = (dist[np.ix_(CORTICAL_INDICES, SUBCORTICAL_INDICES)].sum() + 
                 dist[np.ix_(SUBCORTICAL_INDICES, CORTICAL_INDICES)].sum())
        else:
            t = cm[np.ix_(*idx_pair)].sum()
            d = dist[np.ix_(*idx_pair)].sum()
        
        n = t - d
        print(f"  {label:<33} {int(n):>10,} {int(d):>10,} {int(t):>10,} {pct(n, t):>11.2f}%")

# Overall Reduction Summary
total_all = results["LH"]["cm"].sum() + results["RH"]["cm"].sum()
dist_all = results["LH"]["distal"].sum() + results["RH"]["distal"].sum()
print(f"\n{'='*100}\nFINAL SUMMARY: Removed {int(total_all-dist_all):,} nearby errors ({pct(total_all-dist_all, total_all):.2f}% reduction)\n{'='*100}")

STEP 3: INTEGRATED SPATIAL & ANATOMICAL ANALYSIS

[LH HEMISPHERE] Total Errors: 2,611 | Reduction: 9.73%
----------------------------------------------------------------------------------------------------
Metric                                  Nearby     Distal      Total     Nearby %
From Cortical                               70      1,544      1,614        4.34%
From Subcortical                           184        813        997       18.46%

CONFUSION TYPES (Boundary Analysis)
  Within-Cortical (C<->C)                   48      1,020      1,068        4.49%
  Within-Subcortical (S<->S)               171        325        496       34.48%
  Cross-Group (C<->S)                       35      1,012      1,047        3.34%

[RH HEMISPHERE] Total Errors: 2,791 | Reduction: 9.92%
----------------------------------------------------------------------------------------------------
Metric                                  Nearby     Distal      Total     Nearby %
From Cortical             

In [22]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Data and Variables
errs = [12511, 5406, 4873] # baseline_errors, hemi_errors, spatial_errors
stages = ['Baseline\n(Full)', 'Step 2\n(Hemispheres)', 'Step 3\n(Spatial Filter)']
colors = ['#d62728', '#ff7f0e', '#2ca02c']
true_test_size = len(left_preds) + len(right_preds)
accs = [100 * (1 - e / true_test_size) for e in errs]

fig = make_subplots(rows=1, cols=2, specs=[[{'type': 'bar'}, {'type': 'scatter'}]],
                    subplot_titles=('<b>Total Errors per Stage</b>', '<b>Test Accuracy Progression</b>'))

# Plot 1: Errors
fig.add_trace(go.Bar(x=stages, y=errs, marker_color=colors, text=[f"{x:,}" for x in errs], 
                     textposition='outside', width=0.35, showlegend=False), row=1, col=1)

# Annotations for Reductions
for i in range(2):
    diff = errs[i] - errs[i+1]
    fig.add_annotation(x=i + 0.5, y=(errs[i] + errs[i+1])/2, showarrow=False, row=1, col=1,
                       text=f"↓{diff:,}<br>({diff/errs[i]*100:.1f}%)",
                       font=dict(size=12, color='red', family='Arial Black'),
                       bgcolor='rgba(255,255,255,0.9)', bordercolor='red', borderwidth=1)

# Plot 2: Accuracy
fig.add_trace(go.Scatter(x=stages, y=accs, mode='lines+markers+text', name='Accuracy',
                         marker=dict(size=15, color=colors), line=dict(width=5, color='#1f77b4'),
                         text=[f"{val:.1f}%" for val in accs], textposition='top center',
                         textfont=dict(size=14, family='Arial Black')), row=1, col=2)

# Unified Layout Update
fig.update_xaxes(title_text="Pipeline Stage")
fig.update_yaxes(title_text="Number of Errors", row=1, col=1, rangemode='tozero')
fig.update_yaxes(title_text="Test Accuracy (%)", row=1, col=2, range=[0, 110])

fig.update_layout(title={'text': '<b>Systematic Error Reduction</b>', 'x': 0.5, 'xanchor': 'center', 
                         'font': {'size': 22, 'family': 'Arial Black', 'color': 'darkblue'}},
                  template='plotly_white', height=600, width=1400, showlegend=False, margin=dict(t=120))
fig.show()

In [23]:
# =============================================================================
# PUBLICATION-READY HEMISPHERE COMPARISONS 
# =============================================================================


# 1. Prepare Metadata for Hover Interactions
# Map region names to a 116x116 grid for interactive tooltips
region_names = {
    'LH': lh_region_info['region_name'].tolist(), 
    'RH': rh_region_info['region_name'].tolist()
}

hover_data_lh = [
    [[region_names['LH'][i], region_names['LH'][j]] for j in range(116)] 
    for i in range(116)
]
hover_data_rh = [
    [[region_names['RH'][i], region_names['RH'][j]] for j in range(116)] 
    for i in range(116)
]

# 2. Shared Publication-Quality Configuration
# Calculate global max error to ensure baseline and filtered plots use the same color scale
max_error_count = max(cm_left.max(), cm_right.max())

heatmap_settings = dict(
    x=list(range(1, 117)), 
    y=list(range(1, 117)), 
    zmin=0, 
    zmax=max_error_count,
    colorscale='YlOrRd', 
    hovertemplate=(
        "<b>True:</b> %{y} (%{customdata[0]})<br>"
        "<b>Pred:</b> %{x} (%{customdata[1]})<br>"
        "<b>Errors:</b> %{z}<extra></extra>"
    )
)

def plot_hemisphere_comparison(matrix_left, matrix_right, title, figure_id):
    """Generates a side-by-side heatmap comparison for LH and RH."""
    fig = make_subplots(
        rows=1, cols=2, 
        horizontal_spacing=0.15, 
        subplot_titles=(
            f'<b>{figure_id}1</b> Left Hemisphere', 
            f'<b>{figure_id}2</b> Right Hemisphere'
        )
    )
    
    # Add Heatmap traces
    fig.add_trace(
        go.Heatmap(z=matrix_left, customdata=hover_data_lh, showscale=False, **heatmap_settings), 
        row=1, col=1
    )
    fig.add_trace(
        go.Heatmap(z=matrix_right, customdata=hover_data_rh, showscale=True, **heatmap_settings, 
                   colorbar=dict(title="Error Count", x=1.02, thickness=12, len=0.8)), 
        row=1, col=2
    )
    
    # Global Layout Styling
    fig.update_layout(
        title={'text': title, 'x': 0.5, 'font': {'size': 18, 'family': 'Arial'}},
        template='plotly_white', 
        height=550, 
        width=1000, 
        margin=dict(t=100, b=50)
    )
    
    # FIXED AXES: Strictly 1 to 117
    # constrain='domain' forces Plotly to respect the range and scaleanchor ensures a square plot
    axis_config = dict(
        range=[1, 117], 
        tick0=1, 
        dtick=20, 
        gridcolor='#f0f0f0',
        constrain='domain',
        showline=True,
        linewidth=1,
        linecolor='black',
        mirror=True
    )
    
    fig.update_xaxes(title_text="Predicted Region Index", **axis_config)
    fig.update_yaxes(title_text="True Region Index", scaleanchor="x", **axis_config)
    
    fig.show()

# 3. Execute Visualizations
plot_hemisphere_comparison(
    cm_left, cm_right, 
    "<b>A.</b> Baseline Region-Level Misclassification", "A"
)
plot_hemisphere_comparison(
    cm_left_distal, cm_right_distal, 
    f"<b>B.</b> Distal Misclassification (>{DISTANCE_THRESHOLD}mm Filter)", "B"
)

# 4. Impact Summary: Comprehensive Error Removal Analysis
print(f"{'='*90}\nSPATIAL FILTERING IMPACT SUMMARY: ERROR REDUCTION PER HEMISPHERE\n{'='*90}")
print(f"{'Hemi':<10} | {'Baseline':>10} | {'Filtered':>10} | {'Errors Removed':>18} | {'Reduction %'}")
print("-" * 90)

hemisphere_results = [
    ("Left", cm_left, cm_left_distal), 
    ("Right", cm_right, cm_right_distal)
]

for label, baseline_cm, filtered_cm in hemisphere_results:
    total_baseline = int(np.sum(baseline_cm))
    total_filtered = int(np.sum(filtered_cm))
    removed_count = total_baseline - total_filtered
    reduction_percentage = (removed_count / total_baseline * 100) if total_baseline > 0 else 0
    
    print(f"{label:<10} | {total_baseline:>10,} | {total_filtered:>10,} | {removed_count:>18,} | {reduction_percentage:>10.1f}%")

print(f"{'='*90}")

SPATIAL FILTERING IMPACT SUMMARY: ERROR REDUCTION PER HEMISPHERE
Hemi       |   Baseline |   Filtered |     Errors Removed | Reduction %
------------------------------------------------------------------------------------------
Left       |      2,611 |      2,357 |                254 |        9.7%
Right      |      2,791 |      2,514 |                277 |        9.9%


In [24]:
# =============================================================================
# 1. DATA PROCESSING: MAP 17 NETWORKS -> YEO 7 + SUBCORTICAL
# =============================================================================
def add_yeo7_mapping(df):
    """
    Creates a new column 'yeo7_grouped' by aggregating the 17-network 
    labels into the standard 7 classes + Subcortical.
    """
    def map_row(row):
        # 1. Subcortical Check
        # Based on your data, indices 100-115 are subcortical
        if row['region_idx'] >= 100:
            return 'Subcortical'
        
        # 2. Cortical Mapping (17 -> 7)
        net = row['network']
        
        if 'Vis' in net:          return 'Visual'
        if 'SomMot' in net:       return 'SomMot'
        if 'DorsAttn' in net:     return 'DorsAttn'
        if 'SalVentAttn' in net:  return 'SalVentAttn'
        if 'Limbic' in net:       return 'Limbic'
        if 'Cont' in net:         return 'Cont'
        # TempPar is historically grouped with Default in the 7-network parcellation
        if 'Default' in net or 'TempPar' in net: return 'Default'
        
        return 'Other'

    # Create the new column
    df_copy = df.copy()
    df_copy['yeo7_grouped'] = df_copy.apply(map_row, axis=1)
    return df_copy

# --- APPLY THE MAPPING IMMEDIATELY ---
# We update your dataframes with the new grouping column
lh_region_info = add_yeo7_mapping(lh_region_info)
rh_region_info = add_yeo7_mapping(rh_region_info)

# SET THE CONFIG TO USE THIS NEW GROUPING
GROUPING_COLUMN = 'yeo7_grouped' 

# =============================================================================
# 2. HELPER FUNCTIONS
# =============================================================================
def get_network_boundaries(region_info, network_column):
    # Ensure the column exists
    if network_column not in region_info.columns:
        raise KeyError(f"Column '{network_column}' not found. Run the mapping step first.")

    networks, boundaries = [], [0]
    current_network = None
    
    # Iterate through the DataFrame (assuming it's sorted by region_idx)
    for idx, row in region_info.iterrows():
        network = row[network_column]
        if network != current_network:
            if current_network is not None:
                boundaries.append(idx)
                networks.append(current_network)
            current_network = network
            
    boundaries.append(len(region_info))
    networks.append(current_network)
    
    # Calculate midpoints for label placement
    midpoints = [(boundaries[i] + boundaries[i+1]) / 2 for i in range(len(boundaries)-1)]
    return boundaries, networks, midpoints

def bin_errors_discrete(matrix):
    binned = np.zeros_like(matrix, dtype=float)
    binned[matrix == 0] = np.nan
    binned[(matrix >= 1) & (matrix <= 5)] = 0.5
    binned[(matrix >= 6) & (matrix <= 10)] = 1.5
    binned[(matrix >= 11) & (matrix <= 15)] = 2.5
    binned[(matrix >= 16) & (matrix <= 20)] = 3.5
    binned[matrix > 20] = 4.5
    return binned

# =============================================================================
# 3. PLOTTING FUNCTION
# =============================================================================
def plot_research_heatmap(matrix_left, matrix_right, lh_info, rh_info, title, figure_id):
    
    # --- Data Prep ---
    # We now use the GROUPING_COLUMN ('yeo7_grouped') defined above
    lh_bounds, lh_nets, lh_mids = get_network_boundaries(lh_info, GROUPING_COLUMN)
    rh_bounds, rh_nets, rh_mids = get_network_boundaries(rh_info, GROUPING_COLUMN)
    
    z_lh = bin_errors_discrete(matrix_left)
    z_rh = bin_errors_discrete(matrix_right)
    discrete_colors = ['#FDB462', '#FB8072', '#E31A1C', '#BD0026', '#800026']

    # --- Setup Figure ---
    title_l = f'<b>{figure_id}1.</b> Left Hemisphere'
    title_r = f'<b>{figure_id}2.</b> Right Hemisphere'

    fig = make_subplots(
        rows=1, cols=2, 
        horizontal_spacing=0.12, 
        subplot_titles=(title_l, title_r)
    )

    # --- Heatmaps ---
    h_cfg = dict(
        x=list(range(1, 117)), y=list(range(1, 117)),
        zmin=0, zmax=5, colorscale=discrete_colors,
        xgap=0, ygap=0,
        hovertemplate="True: %{y}<br>Pred: %{x}<br>Errors: %{customdata}<extra></extra>"
    )
    
    fig.add_trace(go.Heatmap(z=z_lh, customdata=matrix_left, showscale=False, **h_cfg), 1, 1)
    
    cbar = dict(
        title=dict(text="<b>Error Count</b>", font=dict(size=12)),
        tickmode="array", tickvals=[0.5, 1.5, 2.5, 3.5, 4.5],
        ticktext=["1-5", "6-10", "11-15", "16-20", ">20"],
        thickness=15, len=0.6, yanchor="middle", xpad=10
    )
    fig.add_trace(go.Heatmap(z=z_rh, customdata=matrix_right, showscale=True, colorbar=cbar, **h_cfg), 1, 2)

    # --- Overlays & Labels ---
    def add_overlays(row, col, bounds, nets, mids):
        # Grid Lines
        for b in bounds[1:-1]:
            line_pos = b + 0.5 
            # Vertical
            fig.add_shape(type="line", x0=line_pos, x1=line_pos, y0=0.5, y1=116.5,
                          line=dict(color="black", width=0.5), row=row, col=col)
            # Horizontal
            fig.add_shape(type="line", x0=0.5, x1=116.5, y0=line_pos, y1=line_pos,
                          line=dict(color="black", width=0.5), row=row, col=col)
        
        # Outer Frame
        fig.add_shape(type="rect", x0=0.5, x1=116.5, y0=0.5, y1=116.5,
                      line=dict(color="black", width=1.5), fillcolor="rgba(0,0,0,0)", row=row, col=col)

        # Network Labels (Rotated)
        for i, net in enumerate(nets):
            fig.add_annotation(
                x=mids[i], y=116.5, yshift=10, 
                text=f"<b>{net}</b>", showarrow=False,
                font=dict(size=9, color="#333333", family="Arial"),
                textangle=-90, xanchor='center', yanchor='bottom',
                row=row, col=col
            )

    add_overlays(1, 1, lh_bounds, lh_nets, lh_mids)
    add_overlays(1, 2, rh_bounds, rh_nets, rh_mids)

    # --- Final Layout Polish ---
    # 1. Shift Subplot Titles Up
    fig.for_each_annotation(lambda a: a.update(yshift=70) if a.text in [title_l, title_r] else None)

    # 2. Main Layout
    fig.update_layout(
        title={'text': title, 'y': 0.98, 'x': 0.5, 'xanchor': 'center', 'yanchor': 'top', 'font': dict(size=18, family="Arial")},
        template='plotly_white',
        height=700, width=1200, 
        font=dict(family="Arial", size=12, color="black"),
        margin=dict(t=160, l=80, r=80, b=60) 
    )

    axis_style = dict(
        showgrid=False, zeroline=False, range=[0.5, 116.5],
        tickmode='array', tickvals=[1, 20, 40, 60, 80, 100, 116], 
        showline=False, ticks="outside", ticklen=5, constrain='domain'
    )
    fig.update_xaxes(title_text="Predicted Region", **axis_style)
    fig.update_yaxes(title_text="True Region", scaleanchor='x', scaleratio=1, **axis_style)
    
    fig.show()

# =============================================================================
# 4. EXECUTION
# =============================================================================
plot_research_heatmap(
    cm_left, cm_right, 
    lh_region_info, rh_region_info, 
    "<b>A. Baseline Misclassification (Yeo-7 Networks)</b>", "A"
)
plot_research_heatmap(
    cm_left_distal, 
    cm_right_distal, 
    lh_region_info, 
    rh_region_info, 
    "<b>B. Distal Misclassification (>20mm Filter)</b>", 
    "B"
)

# 5. Impact Summary
print(f"{'='*90}\nSPATIAL FILTERING IMPACT SUMMARY: ERROR REDUCTION PER HEMISPHERE\n{'='*90}")
print(f"{'Hemi':<10} | {'Baseline':>10} | {'Filtered':>10} | {'Errors Removed':>18} | {'Reduction %'}")
print("-" * 90)

hemisphere_results = [
    ("Left", cm_left, cm_left_distal), 
    ("Right", cm_right, cm_right_distal)
]

for label, baseline_cm, filtered_cm in hemisphere_results:
    total_baseline = int(np.sum(baseline_cm))
    total_filtered = int(np.sum(filtered_cm))
    removed_count = total_baseline - total_filtered
    reduction_percentage = (removed_count / total_baseline * 100) if total_baseline > 0 else 0
    
    print(f"{label:<10} | {total_baseline:>10,} | {total_filtered:>10,} | {removed_count:>18,} | {reduction_percentage:>10.1f}%")

print(f"{'='*90}")

SPATIAL FILTERING IMPACT SUMMARY: ERROR REDUCTION PER HEMISPHERE
Hemi       |   Baseline |   Filtered |     Errors Removed | Reduction %
------------------------------------------------------------------------------------------
Left       |      2,611 |      2,357 |                254 |        9.7%
Right      |      2,791 |      2,514 |                277 |        9.9%


## Network Analysis 